# 单吸附模型构建工具

本工具用于创建各种分子在Cu(100)表面的单吸附模型。

## 功能特点：
- 支持分子：H2O, H2, OH, H, O
- 支持位点：hollow, top, bridge（每个分子一次生成三个位点）
- 对于H2O和H2，支持调整取向（旋转角度）
- 支持调整吸附距离（间距）
- 每个结构保存到独立的文件夹

## 第一步：全局设置（只需设置一次）

In [10]:
import numpy as np
from ase import Atoms
from ase.io import read, write
from pathlib import Path
import math

# ==========================================================
# 全局设置（只需设置一次）
# ==========================================================

# 表面结构文件路径（Cu(100)表面POSCAR）
surface_poscar = "/home/ucaqzh0/thermol/100_water/absorption/absorption/POSCAR"

# 基础输出目录（仿照GO目录结构）
base_output_dir = Path("./absorption_single_models")
base_output_dir.mkdir(exist_ok=True)

# 读取表面结构（只读一次，后续复用）
print("正在读取表面结构...")
surface_atoms = read(surface_poscar, format='vasp')
print(f"✓ 已读取表面结构: {len(surface_atoms)} 原子")
print(f"✓ 表面结构文件: {surface_poscar}")

正在读取表面结构...
✓ 已读取表面结构: 48 原子
✓ 表面结构文件: /home/ucaqzh0/thermol/100_water/absorption/absorption/POSCAR


## 第一步（续）：可视化表面位点位置（可选，用于快速定位）

运行下面的cell可以查看所有高对称位点的位置，确保所有构型都在中心位置且对齐。

In [13]:
# ==========================================================
# 可视化表面位点位置（用于快速定位高对称位点）
# ==========================================================
# 运行此cell可以查看所有位点的坐标，确保它们都在中心位置且对齐
# 这对于NEB计算很重要，因为初始和最终状态需要在空间上对齐

# 注意：需要先运行上面的cell定义visualize_surface_sites函数
# 如果还没有定义，可以先运行"核心函数定义"的cell

try:
    site_positions = visualize_surface_sites(surface_atoms)
    print("✓ 位点位置已显示")
except NameError:
    print("⚠ 请先运行上面的'核心函数定义'cell")

⚠ 请先运行上面的'核心函数定义'cell


## 第二步：核心函数定义（只需运行一次）

In [14]:
# ==========================================================
# 核心函数：计算表面位点坐标（基于晶胞中心，确保NEB计算对齐）
# ==========================================================

def get_surface_site_position(surface_atoms, site='hollow'):
    """
    计算Cu(100)表面的吸附位点坐标
    
    所有位点都基于晶胞中心(a/2, b/2)来定位，确保：
    1. 所有构型都在中心位置
    2. 相同的位置完成生成（便于NEB计算时空间对齐）
    3. 快速定位高对称位点
    """
    cell = surface_atoms.get_cell()
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    
    # 晶胞中心位置（xy平面）
    center_xy = np.array([a/2, b/2])
    
    cu_indices = [i for i, s in enumerate(surface_atoms.get_chemical_symbols()) 
                  if s.upper() == 'CU']
    if not cu_indices:
        raise ValueError("未找到Cu原子")
    
    cu_positions = surface_atoms.get_positions()[cu_indices]
    top_z = np.max(cu_positions[:, 2])
    top_cu_positions = cu_positions[cu_positions[:, 2] > top_z - 0.1]
    
    if site == 'hollow':
        # Hollow位点：四个Cu原子的中心，正好在(a/2, b/2)
        site_pos = np.array([center_xy[0], center_xy[1], top_z])
        
    elif site == 'top':
        # Top位点：找到最接近晶胞中心的Cu原子
        # 如果有多个相同距离的候选，优先选择右上方（x和y更大）
        distances_to_center = np.linalg.norm(top_cu_positions[:, :2] - center_xy, axis=1)
        min_distance = np.min(distances_to_center)
        # 找到所有距离最小的候选（容差1e-6）
        candidates = np.where(distances_to_center < min_distance + 1e-6)[0]
        if len(candidates) > 1:
            # 优先选择右上方：x和y坐标更大的
            candidate_positions = top_cu_positions[candidates]
            # 计算 x + y 的和，选择最大的
            xy_sum = candidate_positions[:, 0] + candidate_positions[:, 1]
            best_idx_in_candidates = np.argmax(xy_sum)
            nearest_idx = candidates[best_idx_in_candidates]
        else:
            nearest_idx = candidates[0]
        site_pos = top_cu_positions[nearest_idx].copy()
        site_pos[2] = top_z
        
    elif site == 'bridge':
        # Bridge位点：找到最接近晶胞中心的两个相邻Cu原子的中点
        # 首先找到最接近中心的Cu原子（优先右上方）
        distances_to_center = np.linalg.norm(top_cu_positions[:, :2] - center_xy, axis=1)
        min_distance = np.min(distances_to_center)
        candidates = np.where(distances_to_center < min_distance + 1e-6)[0]
        if len(candidates) > 1:
            # 优先选择右上方：x和y坐标更大的
            candidate_positions = top_cu_positions[candidates]
            xy_sum = candidate_positions[:, 0] + candidate_positions[:, 1]
            best_idx_in_candidates = np.argmax(xy_sum)
            nearest_idx = candidates[best_idx_in_candidates]
        else:
            nearest_idx = candidates[0]
        p1 = top_cu_positions[nearest_idx]
        
        # 找到p1的最近邻（距离约为a/2或b/2）
        distances_to_p1 = np.linalg.norm(top_cu_positions[:, :2] - p1[:2], axis=1)
        # 排除自己
        distances_to_p1[nearest_idx] = np.inf
        # 寻找距离约为a/2的邻居（最近邻）
        neighbor_distances = np.abs(distances_to_p1 - a/2)
        min_neighbor_distance = np.min(neighbor_distances)
        if min_neighbor_distance < 0.3:  # 容差0.3 Å
            # 如果有多个相同距离的邻居，优先选择右上方
            neighbor_candidates = np.where(neighbor_distances < min_neighbor_distance + 1e-6)[0]
            if len(neighbor_candidates) > 1:
                neighbor_positions = top_cu_positions[neighbor_candidates]
                # 选择使得中点更靠右上方的邻居（即p2的x+y更大）
                xy_sum_neighbors = neighbor_positions[:, 0] + neighbor_positions[:, 1]
                best_neighbor_idx_in_candidates = np.argmax(xy_sum_neighbors)
                neighbor_idx = neighbor_candidates[best_neighbor_idx_in_candidates]
            else:
                neighbor_idx = neighbor_candidates[0]
            p2 = top_cu_positions[neighbor_idx]
            site_pos = (p1 + p2) / 2
            site_pos[2] = top_z
        else:
            # 如果找不到合适的邻居，使用理论位置
            # Bridge位点应该在(a/2, b/4)或(a/4, b/2)，优先选择右上方
            bridge_candidates = [
                np.array([center_xy[0], center_xy[1] - a/4, top_z]),  # 下方
                np.array([center_xy[0] - a/4, center_xy[1], top_z]),  # 左方
                np.array([center_xy[0], center_xy[1] + a/4, top_z]),  # 上方
                np.array([center_xy[0] + a/4, center_xy[1], top_z]),  # 右方
            ]
            # 优先选择右上方：x和y坐标更大的
            xy_sums = [cand[0] + cand[1] for cand in bridge_candidates]
            site_pos = bridge_candidates[np.argmax(xy_sums)]
    else:
        raise ValueError(f"未知的位点类型: {site}")
    
    return site_pos


def visualize_surface_sites(surface_atoms):
    """
    可视化表面位点位置，帮助快速定位高对称位点
    
    返回所有位点的坐标信息
    """
    sites = ['hollow', 'top', 'bridge']
    site_positions = {}
    
    cell = surface_atoms.get_cell()
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    center_xy = np.array([a/2, b/2])
    
    print(f"\n{'='*70}")
    print(f"表面位点位置信息（基于晶胞中心对齐）")
    print(f"{'='*70}")
    print(f"晶胞参数: a = {a:.6f} Å, b = {b:.6f} Å")
    print(f"晶胞中心 (xy): ({center_xy[0]:.6f}, {center_xy[1]:.6f}) Å")
    print(f"\n位点坐标（笛卡尔坐标）:")
    
    for site in sites:
        pos = get_surface_site_position(surface_atoms, site=site)
        site_positions[site] = pos
        distance_from_center = np.linalg.norm(pos[:2] - center_xy)
        print(f"  {site:8s}: ({pos[0]:10.6f}, {pos[1]:10.6f}, {pos[2]:10.6f}) Å  "
              f"[距离中心: {distance_from_center:.6f} Å]")
    
    print(f"{'='*70}\n")
    
    return site_positions


# ==========================================================
# 核心函数：创建分子结构
# ==========================================================

def create_molecule(molecule_type):
    """创建分子结构（在原点，未旋转）"""
    if molecule_type == 'H':
        molecule = Atoms('H', positions=[[0, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'O':
        molecule = Atoms('O', positions=[[0, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'OH':
        molecule = Atoms('OH', positions=[[0, 0, 0], [0, 0, 0.97]])
        anchor_idx = 0
    elif molecule_type == 'H2':
        molecule = Atoms('HH', positions=[[0, 0, 0], [0.74, 0, 0]])
        anchor_idx = 0
    elif molecule_type == 'H2O':
        # H2O分子结构：O-H键长0.96 Å，H-O-H键角104.5度
        oh_length = 0.96  # O-H键长（Å）
        hoh_angle_deg = 104.5  # H-O-H键角（度）
        hoh_angle_rad = hoh_angle_deg * np.pi / 180
        
        # O原子在原点
        o_pos = np.array([0, 0, 0])
        
        # H1和H2关于y轴对称，都在x轴正方向一侧
        # H1在角度为 hoh_angle/2 的位置
        # H2在角度为 -hoh_angle/2 的位置
        half_angle = hoh_angle_rad / 2
        h1_pos = np.array([oh_length * np.cos(half_angle), oh_length * np.sin(half_angle), 0])
        h2_pos = np.array([oh_length * np.cos(half_angle), -oh_length * np.sin(half_angle), 0])
        
        molecule = Atoms('OHH', positions=[o_pos, h1_pos, h2_pos])
        anchor_idx = 0  # O原子作为锚点
    else:
        raise ValueError(f"不支持的分子类型: {molecule_type}")
    
    return molecule, anchor_idx


# ==========================================================
# 核心函数：旋转分子（用于调整取向）
# ==========================================================

def rotation_matrix_from_vectors(v1, v2):
    """
    计算将向量v1旋转到v2的旋转矩阵
    
    参数:
        v1: 初始向量（3D数组）
        v2: 目标向量（3D数组）
    
    返回:
        rotation_matrix: 3x3旋转矩阵
    """
    v1 = np.array(v1, dtype=float)
    v2 = np.array(v2, dtype=float)
    
    # 归一化
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    
    # 如果向量相同，返回单位矩阵
    if np.allclose(v1, v2):
        return np.eye(3)
    
    # 如果向量相反，需要特殊处理
    if np.allclose(v1, -v2):
        # 找一个垂直于v1的向量
        if abs(v1[0]) < 0.9:
            perp = np.array([1, 0, 0])
        else:
            perp = np.array([0, 1, 0])
        perp = perp - np.dot(perp, v1) * v1
        perp = perp / np.linalg.norm(perp)
        # 旋转180度
        return 2 * np.outer(v1, v1) - np.eye(3)
    
    # 使用Rodrigues旋转公式
    v = np.cross(v1, v2)
    s = np.linalg.norm(v)
    c = np.dot(v1, v2)
    
    vx = np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])
    
    rotation = np.eye(3) + vx + vx @ vx * ((1 - c) / (s ** 2))
    
    return rotation


def rotate_molecule(molecule, anchor_idx, molecule_type, 
                   direction_vector=None, normal_vector=None):
    """
    旋转分子以调整取向（使用向量方式）
    只旋转分子，不改变分子的内部几何结构
    
    参数:
        molecule: ASE Atoms对象
        anchor_idx: 锚定原子索引
        molecule_type: 分子类型
        direction_vector: 方向向量（3D数组），用于H2指定H-H键的方向
        normal_vector: 法向量（3D数组），用于H2O指定分子平面的法向量
    """
    molecule = molecule.copy()
    anchor_pos = molecule.get_positions()[anchor_idx]
    positions = molecule.get_positions() - anchor_pos
    molecule.set_positions(positions)
    
    if molecule_type == 'H2' and direction_vector is not None:
        # H2: 使用方向向量指定H-H键方向
        # 默认H-H键沿x轴 [1, 0, 0]
        default_direction = np.array([1, 0, 0])
        rotation = rotation_matrix_from_vectors(default_direction, direction_vector)
    elif molecule_type == 'H2O' and normal_vector is not None:
        # H2O: 使用法向量指定分子平面的法向量
        # 默认H2O在xy平面，法向量沿z轴 [0, 0, 1]
        default_normal = np.array([0, 0, 1])
        rotation = rotation_matrix_from_vectors(default_normal, normal_vector)
    else:
        # 不旋转
        rotation = np.eye(3)
    
    positions = molecule.get_positions()
    rotated_positions = positions @ rotation.T
    molecule.set_positions(rotated_positions)
    
    return molecule


# ==========================================================
# 核心函数：创建吸附结构
# ==========================================================

def create_adsorption_structure(surface_atoms, molecule_type, adsorption_site, 
                                adsorption_distance, 
                                direction_vector=None, normal_vector=None):
    """
    创建吸附结构
    
    参数:
        direction_vector: 方向向量（3D数组），用于H2指定H-H键方向
        normal_vector: 法向量（3D数组），用于H2O指定分子平面的法向量
    """
    site_pos = get_surface_site_position(surface_atoms, site=adsorption_site)
    molecule, anchor_idx = create_molecule(molecule_type)
    
    # 旋转分子（只改变取向，不改变几何结构）
    if molecule_type == 'H2':
        molecule = rotate_molecule(molecule, anchor_idx, molecule_type, 
                                   direction_vector=direction_vector)
    elif molecule_type == 'H2O':
        molecule = rotate_molecule(molecule, anchor_idx, molecule_type, 
                                   normal_vector=normal_vector)
    
    mol_positions = molecule.get_positions()
    if molecule_type in ['H2O', 'H2']:
        mol_center = np.mean(mol_positions, axis=0)
    else:
        mol_center = mol_positions[anchor_idx]
    
    target_pos = site_pos.copy()
    target_pos[2] += adsorption_distance
    
    anchor_pos = mol_positions[anchor_idx]
    if molecule_type in ['H2O', 'H2']:
        translation = target_pos - mol_center
    else:
        translation = target_pos - anchor_pos
    
    mol_positions = molecule.get_positions() + translation
    molecule.set_positions(mol_positions)
    
    final_atoms = surface_atoms.copy()
    final_atoms.extend(molecule)
    
    return final_atoms

print("✓ 核心函数已定义")

✓ 核心函数已定义


## 第三步：生成各个吸附结构

每种分子一次生成三个位点（hollow, top, bridge），通过 `enable` 开关控制是否生成。

### H2O（水分子）- 三个位点

In [15]:
# ==========================================================
# H2O @ Hollow/Top/Bridge位点
# ==========================================================

molecule_type = 'H2O'
adsorption_distance = 2.5  # Å

# H2O取向参数（向量方式）
# h2o_normal_vector: 分子平面的法向量（垂直于分子平面的方向），例如：
#   [0, 0, 1]  - 分子平面平行于表面（平躺）
#   [0, 0, -1] - 分子平面平行于表面，法向量向下
#   [1, 0, 0]  - 分子平面垂直于表面，法向量沿x方向
#   [0, 1, 0]  - 分子平面垂直于表面，法向量沿y方向
#   [1, 1, 0]  - 法向量在对角线方向（xy平面内）
#   [1, 1, 1]  - 法向量在三维空间对角线方向
#   None       - 不旋转（使用默认方向）
h2o_normal_vector = [0, 0, 1]  # 对角线方向（法向量）

# 三个位点的独立开关
enable_hollow = True   # Hollow位点开关
enable_top = True      # Top位点开关
enable_bridge = True   # Bridge位点开关

# ==========================================================
# 显示初始分子结构
# ==========================================================
molecule, anchor_idx = create_molecule(molecule_type)
print(f"\n{'='*60}")
print(f"{molecule_type} 初始分子结构:")
print(f"  原子数: {len(molecule)}")
print(f"  原子类型: {molecule.get_chemical_symbols()}")
print(f"  初始位置:")
for i, (symbol, pos) in enumerate(zip(molecule.get_chemical_symbols(), molecule.get_positions())):
    print(f"    {symbol}{i+1}: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}) Å")
print(f"{'='*60}")

# ==========================================================
# 生成结构（仿照GO目录架构和命名规则）
# ==========================================================

# 创建分子主目录：absorption_single_{molecule_type}
molecule_dir = base_output_dir / f"absorption_single_{molecule_type}"
molecule_dir.mkdir(exist_ok=True)

created_jobs = []  # 记录创建的job目录

# Hollow位点
if enable_hollow:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'hollow', adsorption_distance,
            normal_vector=h2o_normal_vector
        )
        site = 'hollow'
        # 命名规则：job_POSCAR_{molecule_type}_{site}
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        # 创建job目录
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        # 在job目录中保存POSCAR（文件名就是POSCAR）
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        # 在根目录保存POSCAR文件（用于参考）
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ hollow -> {job_dir}")
    except Exception as e:
        print(f"❌ hollow 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 hollow")

# Top位点
if enable_top:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'top', adsorption_distance,
            normal_vector=h2o_normal_vector
        )
        site = 'top'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ top    -> {job_dir}")
    except Exception as e:
        print(f"❌ top 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 top")

# Bridge位点
if enable_bridge:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'bridge', adsorption_distance,
            normal_vector=h2o_normal_vector
        )
        site = 'bridge'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ bridge -> {job_dir}")
    except Exception as e:
        print(f"❌ bridge 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 bridge")

print(f"\n✓ 完成！共生成 {len(created_jobs)} 个结构")
print(f"✓ 输出目录: {molecule_dir}")


H2O 初始分子结构:
  原子数: 3
  原子类型: ['O', 'H', 'H']
  初始位置:
    O1: (0.0000, 0.0000, 0.0000) Å
    H2: (0.5877, 0.7591, 0.0000) Å
    H3: (0.5877, -0.7591, 0.0000) Å
✓ hollow -> absorption_single_models/absorption_single_H2O/job_POSCAR_H2O_hollow
✓ top    -> absorption_single_models/absorption_single_H2O/job_POSCAR_H2O_top
✓ bridge -> absorption_single_models/absorption_single_H2O/job_POSCAR_H2O_bridge

✓ 完成！共生成 3 个结构
✓ 输出目录: absorption_single_models/absorption_single_H2O


### H2（氢气分子）- 三个位点

In [ ]:
# ==========================================================
# H2 @ Hollow/Top/Bridge位点
# ==========================================================

molecule_type = 'H2'
adsorption_distance = 2.5  # Å

# H2取向参数（向量方式）
# h2_direction_vector: H-H键方向向量，例如：
#   [1, 0, 0]  - H-H键沿x方向（水平）
#   [0, 1, 0]  - H-H键沿y方向（水平）
#   [0, 0, 1]  - H-H键垂直向上
#   [0, 0, -1] - H-H键垂直向下
#   [1, 1, 0]  - H-H键在对角线方向（水平）
#   None       - 不旋转（使用默认方向）
h2_direction_vector = None  # 设置为None不旋转，否则使用向量指定方向

# 三个位点的独立开关
enable_hollow = True   # Hollow位点开关
enable_top = True      # Top位点开关
enable_bridge = True   # Bridge位点开关

# ==========================================================
# 显示初始分子结构
# ==========================================================
molecule, anchor_idx = create_molecule(molecule_type)
print(f"\n{'='*60}")
print(f"{molecule_type} 初始分子结构:")
print(f"  原子数: {len(molecule)}")
print(f"  原子类型: {molecule.get_chemical_symbols()}")
print(f"  初始位置:")
for i, (symbol, pos) in enumerate(zip(molecule.get_chemical_symbols(), molecule.get_positions())):
    print(f"    {symbol}{i+1}: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}) Å")
print(f"{'='*60}")

# ==========================================================
# 生成结构（仿照GO目录架构和命名规则）
# ==========================================================

# 创建分子主目录：absorption_single_{molecule_type}
molecule_dir = base_output_dir / f"absorption_single_{molecule_type}"
molecule_dir.mkdir(exist_ok=True)

created_jobs = []  # 记录创建的job目录

# Hollow位点
if enable_hollow:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'hollow', adsorption_distance,
            direction_vector=h2_direction_vector
        )
        site = 'hollow'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ hollow -> {job_dir}")
    except Exception as e:
        print(f"❌ hollow 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 hollow")

# Top位点
if enable_top:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'top', adsorption_distance,
            direction_vector=h2_direction_vector
        )
        site = 'top'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ top    -> {job_dir}")
    except Exception as e:
        print(f"❌ top 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 top")

# Bridge位点
if enable_bridge:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'bridge', adsorption_distance,
            direction_vector=h2_direction_vector
        )
        site = 'bridge'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ bridge -> {job_dir}")
    except Exception as e:
        print(f"❌ bridge 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 bridge")

print(f"\n✓ 完成！共生成 {len(created_jobs)} 个结构")
print(f"✓ 输出目录: {molecule_dir}")

### OH（羟基）- 三个位点

In [ ]:
# ==========================================================
# OH @ Hollow/Top/Bridge位点
# ==========================================================

molecule_type = 'OH'
adsorption_distance = 2.5  # Å

# 三个位点的独立开关
enable_hollow = True   # Hollow位点开关
enable_top = True      # Top位点开关
enable_bridge = True   # Bridge位点开关

# ==========================================================
# 显示初始分子结构
# ==========================================================
molecule, anchor_idx = create_molecule(molecule_type)
print(f"\n{'='*60}")
print(f"{molecule_type} 初始分子结构:")
print(f"  原子数: {len(molecule)}")
print(f"  原子类型: {molecule.get_chemical_symbols()}")
print(f"  初始位置:")
for i, (symbol, pos) in enumerate(zip(molecule.get_chemical_symbols(), molecule.get_positions())):
    print(f"    {symbol}{i+1}: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}) Å")
print(f"{'='*60}")

# ==========================================================
# 生成结构（仿照GO目录架构和命名规则）
# ==========================================================

# 创建分子主目录：absorption_single_{molecule_type}
molecule_dir = base_output_dir / f"absorption_single_{molecule_type}"
molecule_dir.mkdir(exist_ok=True)

created_jobs = []  # 记录创建的job目录

# Hollow位点
if enable_hollow:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'hollow', adsorption_distance
        )
        site = 'hollow'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ hollow -> {job_dir}")
    except Exception as e:
        print(f"❌ hollow 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 hollow")

# Top位点
if enable_top:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'top', adsorption_distance
        )
        site = 'top'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ top    -> {job_dir}")
    except Exception as e:
        print(f"❌ top 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 top")

# Bridge位点
if enable_bridge:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'bridge', adsorption_distance
        )
        site = 'bridge'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ bridge -> {job_dir}")
    except Exception as e:
        print(f"❌ bridge 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 bridge")

print(f"\n✓ 完成！共生成 {len(created_jobs)} 个结构")
print(f"✓ 输出目录: {molecule_dir}")

### H（氢原子）- 三个位点

In [ ]:
# ==========================================================
# H @ Hollow/Top/Bridge位点
# ==========================================================

molecule_type = 'H'
adsorption_distance = 2.5  # Å

# 三个位点的独立开关
enable_hollow = True   # Hollow位点开关
enable_top = True      # Top位点开关
enable_bridge = True   # Bridge位点开关

# ==========================================================
# 显示初始分子结构
# ==========================================================
molecule, anchor_idx = create_molecule(molecule_type)
print(f"\n{'='*60}")
print(f"{molecule_type} 初始分子结构:")
print(f"  原子数: {len(molecule)}")
print(f"  原子类型: {molecule.get_chemical_symbols()}")
print(f"  初始位置:")
for i, (symbol, pos) in enumerate(zip(molecule.get_chemical_symbols(), molecule.get_positions())):
    print(f"    {symbol}{i+1}: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}) Å")
print(f"{'='*60}")

# ==========================================================
# 生成结构（仿照GO目录架构和命名规则）
# ==========================================================

# 创建分子主目录：absorption_single_{molecule_type}
molecule_dir = base_output_dir / f"absorption_single_{molecule_type}"
molecule_dir.mkdir(exist_ok=True)

created_jobs = []  # 记录创建的job目录

# Hollow位点
if enable_hollow:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'hollow', adsorption_distance
        )
        site = 'hollow'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ hollow -> {job_dir}")
    except Exception as e:
        print(f"❌ hollow 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 hollow")

# Top位点
if enable_top:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'top', adsorption_distance
        )
        site = 'top'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ top    -> {job_dir}")
    except Exception as e:
        print(f"❌ top 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 top")

# Bridge位点
if enable_bridge:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'bridge', adsorption_distance
        )
        site = 'bridge'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ bridge -> {job_dir}")
    except Exception as e:
        print(f"❌ bridge 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 bridge")

print(f"\n✓ 完成！共生成 {len(created_jobs)} 个结构")
print(f"✓ 输出目录: {molecule_dir}")

### O（氧原子）- 三个位点

In [ ]:
# ==========================================================
# O @ Hollow/Top/Bridge位点
# ==========================================================

molecule_type = 'O'
adsorption_distance = 2.5  # Å

# 三个位点的独立开关
enable_hollow = True   # Hollow位点开关
enable_top = True      # Top位点开关
enable_bridge = True   # Bridge位点开关

# ==========================================================
# 显示初始分子结构
# ==========================================================
molecule, anchor_idx = create_molecule(molecule_type)
print(f"\n{'='*60}")
print(f"{molecule_type} 初始分子结构:")
print(f"  原子数: {len(molecule)}")
print(f"  原子类型: {molecule.get_chemical_symbols()}")
print(f"  初始位置:")
for i, (symbol, pos) in enumerate(zip(molecule.get_chemical_symbols(), molecule.get_positions())):
    print(f"    {symbol}{i+1}: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}) Å")
print(f"{'='*60}")

# ==========================================================
# 生成结构（仿照GO目录架构和命名规则）
# ==========================================================

# 创建分子主目录：absorption_single_{molecule_type}
molecule_dir = base_output_dir / f"absorption_single_{molecule_type}"
molecule_dir.mkdir(exist_ok=True)

created_jobs = []  # 记录创建的job目录

# Hollow位点
if enable_hollow:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'hollow', adsorption_distance
        )
        site = 'hollow'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ hollow -> {job_dir}")
    except Exception as e:
        print(f"❌ hollow 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 hollow")

# Top位点
if enable_top:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'top', adsorption_distance
        )
        site = 'top'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ top    -> {job_dir}")
    except Exception as e:
        print(f"❌ top 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 top")

# Bridge位点
if enable_bridge:
    try:
        final_atoms = create_adsorption_structure(
            surface_atoms, molecule_type, 'bridge', adsorption_distance
        )
        site = 'bridge'
        job_name = f"job_POSCAR_{molecule_type}_{site}"
        poscar_name = f"POSCAR_{molecule_type}_{site}.vasp"
        
        job_dir = molecule_dir / job_name
        job_dir.mkdir(exist_ok=True)
        
        job_poscar = job_dir / "POSCAR"
        write(str(job_poscar), final_atoms, format='vasp', vasp5=True)
        
        root_poscar = molecule_dir / poscar_name
        write(str(root_poscar), final_atoms, format='vasp', vasp5=True)
        
        created_jobs.append(job_dir)
        print(f"✓ bridge -> {job_dir}")
    except Exception as e:
        print(f"❌ bridge 错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⏭ 跳过 bridge")

print(f"\n✓ 完成！共生成 {len(created_jobs)} 个结构")
print(f"✓ 输出目录: {molecule_dir}")

## 说明

### 如何调节取向？

使用**向量方式**调节取向，直接指定方向向量即可。**只旋转分子，不改变分子的内部几何结构**。

#### 对于H2O（水分子）：

- `h2o_normal_vector`: 分子平面的法向量（垂直于分子平面的方向，3D数组）
  - `[0, 0, 1]` - 分子平面平行于表面（平躺）
  - `[0, 0, -1]` - 分子平面平行于表面，法向量向下
  - `[1, 0, 0]` - 分子平面垂直于表面，法向量沿x方向
  - `[0, 1, 0]` - 分子平面垂直于表面，法向量沿y方向
  - `[1, 1, 0]` - 法向量在对角线方向（xy平面内）
  - `[1, 1, 1]` - 法向量在三维空间对角线方向
  - `None` - 不旋转（使用默认方向）

**示例：**
```python
# 分子平躺
h2o_normal_vector = [0, 0, 1]

# 法向量在对角线方向
h2o_normal_vector = [1, 1, 0]

# 不旋转
h2o_normal_vector = None
```

#### 对于H2（氢气分子）：

- `h2_direction_vector`: H-H键方向向量（3D数组）
  - `[1, 0, 0]` - H-H键沿x方向（水平）
  - `[0, 1, 0]` - H-H键沿y方向（水平）
  - `[0, 0, 1]` - H-H键垂直向上
  - `[0, 0, -1]` - H-H键垂直向下
  - `[1, 1, 0]` - H-H键在对角线方向（水平）
  - `None` - 不旋转（使用默认方向）

**示例：**
```python
# H-H键垂直向上
h2_direction_vector = [0, 0, 1]

# H-H键水平沿x方向
h2_direction_vector = [1, 0, 0]

# 不旋转
h2_direction_vector = None
```

### 使用方法：
1. 在第一个cell中设置 `surface_poscar` 路径（只需设置一次）
2. 运行第一个和第二个cell加载函数定义
3. 在每个分子的cell中：
   - **显示初始结构**：运行cell时会自动显示分子的初始结构信息（原子数、类型、位置）
   - **独立开关控制**：每个位点有独立的开关
     - `enable_hollow = True/False` 控制hollow位点
     - `enable_top = True/False` 控制top位点
     - `enable_bridge = True/False` 控制bridge位点
   - 调整参数（吸附距离、取向角度等）
   - 运行cell生成选中的位点结构
4. 每个结构会保存到独立的文件夹中，文件夹名包含所有参数信息